In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

In [2]:
df = pd.read_csv(r"E:\MLOps\MMT\Data\Cleaned\MMT_Cleaned.csv")

In [3]:
df["Travel Date"].tail(20)

27422                24-09-2019
27423                18-10-2019
27424                25-09-2019
27425                22-10-2019
27426                27-09-2019
27427                23-09-2019
27428                01-10-2019
27429                16-10-2019
27430                30-09-2019
27431                12-10-2019
27432                10-10-2019
27433                28-09-2019
27434    Jaipur|Jodhpur|Udaipur
27435                01-10-2019
27436                 Dalhousie
27437                09-10-2019
27438                24-10-2019
27439                02-10-2019
27440                08-10-2019
27441                10-10-2019
Name: Travel Date, dtype: object

In [4]:
#converted the date colum to check how many data were not date rather 
#were names of places

df["Travel Date"] = pd.to_datetime(
    df["Travel Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

In [5]:
#Checking how many data where not date and were converted to NaT

df["Travel Date"].isna().sum()

np.int64(1087)

In [6]:
#Extracted exact year, month and day for better analysis and feature engineering

df["Travel_Year"] = df["Travel Date"].dt.year
df["Travel_Month"] = df["Travel Date"].dt.month
df["Travel_DayOfWeek"] = df["Travel Date"].dt.dayofweek

In [7]:
#Extracted hotel ratings from the hotel details column using regex

df["Hotel Ratings"] = df["Hotel Details"].str.extract(r":(\d+\.\d+)")

In [8]:
#converting it into numeic

df["Hotel Ratings"] = pd.to_numeric(
    df["Hotel Ratings"],
    errors="coerce")

In [9]:
df["Hotel Ratings"]= df["Hotel Ratings"].fillna(
    df["Hotel Ratings"].mode()[0]
)

In [10]:
df["Travel_Year"].isna().sum()
df["Travel_Month"].isna().sum()
df["Travel_DayOfWeek"].isna().sum()

np.int64(1087)

In [11]:
#Filled it with mode.

df["Travel_Year"] = df["Travel_Year"].fillna(
    df["Travel_Year"].mode()[0]
)

df["Travel_Month"] = df["Travel_Month"].fillna(
    df["Travel_Month"].mode()[0]
)

df["Travel_DayOfWeek"] = df["Travel_DayOfWeek"].fillna(
    df["Travel_DayOfWeek"].mode()[0]
)

In [12]:
#We have eliminated all non date values

df["Travel_Year"].isna().sum()
df["Travel_Month"].isna().sum()
df["Travel_DayOfWeek"].isna().sum()

np.int64(0)

In [13]:
df.to_csv(r"E:\MLOps\MMT\Data\Feature Engineered\MMT_Featur_Eng2.csv", index = False)

In [14]:
##one hot encoding for categorical variables

df = pd.get_dummies(
    df,
    columns=["Package Type","Start City"],
    dtype=int
)

In [15]:
df.columns

Index(['Package Name', 'Destination', 'Itinerary', 'Places Covered',
       'Travel Date', 'Hotel Details', 'Airline', 'Per Person Price',
       'Sightseeing Places Covered', 'Cancellation Rules', 'Travel_Year',
       'Travel_Month', 'Travel_DayOfWeek', 'Hotel Ratings',
       'Package Type_Budget', 'Package Type_Deluxe', 'Package Type_Luxury',
       'Package Type_Premium', 'Package Type_Standard', 'Start City_Mumbai',
       'Start City_New Delhi'],
      dtype='object')

In [16]:
#Here we will work on text data which our model wont understand,
#thats why we use sentence transformer to convert text into numbers/vectors
#which represent the meaning of the text and can be used for model training. 

text_columns = [
    'Package Name',
    'Destination',
    'Itinerary',
    'Places Covered',
    'Hotel Details',
    'Airline',
    'Sightseeing Places Covered',
    'Cancellation Rules'
]

In [17]:
#is a pretrained NLP model that will convert your text columns into numerical 
#embeddings that the ML model can use.

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [19]:
#This creates one embedding column 
#for each of your eight text columns and handles missing values.
from tqdm.auto import tqdm

for column in tqdm(text_columns, desc="Generating embeddings"):
    df[column + '_embedding'] = df[column].fillna("").astype(str).apply(
        lambda text: model.encode(text)
    )

Generating embeddings:   0%|          | 0/8 [00:00<?, ?it/s]

In [37]:
#Removing outliers from the numerical columns using IQR method

Q1 = df["Per Person Price"].quantile(0.25)
Q3 = df["Per Person Price"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df = df[
    df["Per Person Price"].between(lower, upper)
]

print("Original rows:", len(df))
print("Rows after removing outliers:", len(df_no_outliers))
print("Rows removed:", len(df) - len(df_no_outliers))
print("Price range:", lower, "to", upper)

Original rows: 26461
Rows after removing outliers: 26461
Rows removed: 0
Price range: -6311.125 to 43263.875


In [39]:
#After removing outliers we can check the difference in skewness of our target variable.

print("skew:", df["Per Person Price"].skew())

skew: 0.5936431542353465


In [40]:
# Select numerical features for model training
num_features = [
    'Package Type_Standard',
    'Package Type_Premium',
    'Package Type_Luxury',
    'Travel_Month',
    'Package Type_Budget',
    'Package Type_Deluxe',
    'Hotel Ratings',
    'Start City_New Delhi',
    'Start City_Mumbai',
    'Travel_DayOfWeek',
    'Travel_Year'
]

print("Number of numerical features:", len(num_features))

Number of numerical features: 11


In [41]:
# Select the target variable to be predicted by the model
y = df['Per Person Price']

print(y.shape)

(26461,)


In [42]:
# Split the filtered dataset into 80% training and 20% testing data
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print("Training data shape:", train_df.shape)
print("Testing data shape:", test_df.shape)

Training data shape: (21168, 29)
Testing data shape: (5293, 29)


In [43]:
# Initialize PCA with 23 components for dimensionality reduction
from sklearn.decomposition import PCA

n_components = 23

In [ ]:
# Apply PCA separately to each text embedding column
import numpy as np

text_embeddings_pca_train = np.zeros((len(train_df), n_components * len(text_columns)))
text_embeddings_pca_test = np.zeros((len(test_df), n_components * len(text_columns)))

pca_models = {}

for i, col in enumerate(text_columns):
    pca = PCA(n_components=n_components)

    train_embeddings = np.stack(train_df[col + '_embedding'].values)
    test_embeddings = np.stack(test_df[col + '_embedding'].values)

    start = i * n_components
    end = start + n_components

    text_embeddings_pca_train[:, start:end] = pca.fit_transform(train_embeddings)
    text_embeddings_pca_test[:, start:end] = pca.transform(test_embeddings)

    pca_models[col] = pca

print("PCA training shape:", text_embeddings_pca_train.shape)
print("PCA testing shape:", text_embeddings_pca_test.shape)

PCA training shape: (21168, 184)
PCA testing shape: (5293, 184)


In [45]:
# Select numerical features for training and testing
x_train_numerical = train_df[num_features]
x_test_numerical = test_df[num_features]

# Select target variable for training and testing
y_train = train_df['Per Person Price']
y_test = test_df['Per Person Price']

print("Numerical training shape:", x_train_numerical.shape)
print("Numerical testing shape:", x_test_numerical.shape)
print("Target training shape:", y_train.shape)
print("Target testing shape:", y_test.shape)

Numerical training shape: (21168, 11)
Numerical testing shape: (5293, 11)
Target training shape: (21168,)
Target testing shape: (5293,)


In [46]:
# Combine PCA-reduced text embeddings with numerical features

x_train = pd.concat(
    [
        pd.DataFrame(text_embeddings_pca_train, index=train_df.index),
        x_train_numerical
    ],
    axis=1
)

x_test = pd.concat(
    [
        pd.DataFrame(text_embeddings_pca_test, index=test_df.index),
        x_test_numerical
    ],
    axis=1
)

# Check the final feature shapes
print("X_train shape:", x_train.shape)
print("X_test shape:", x_test.shape)

X_train shape: (21168, 195)
X_test shape: (5293, 195)


In [49]:

import os
import joblib


output_path = r"E:\MLOps\MMT\Data\Feature Engineered"
os.makedirs(output_path, exist_ok=True)

# Save all eight fitted PCA models
joblib.dump(pca_models, os.path.join(output_path, "pca_models.joblib"))

['E:\\MLOps\\MMT\\Data\\Feature Engineered\\pca_models.joblib']

In [50]:
# Save training and testing features

output_path = r"E:\MLOps\MMT\Data\Training Data"

x_train.to_csv(os.path.join(output_path, "x_train.csv"), index=False)
x_test.to_csv(os.path.join(output_path, "x_test.csv"), index=False)

# Save training and testing target values
y_train.to_csv(os.path.join(output_path, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(output_path, "y_test.csv"), index=False)

print("PCA models and train/test datasets saved successfully!")

PCA models and train/test datasets saved successfully!
